# Sharding

In this demo and exercise, we explore the effects of sharding on Zarr arrays. Again, we use the two-photon data we wrote to a Zarr file on Monday.

## Chunks-only first

We start by lazily opening the two-photon data (in read-only mode, `mode="r"`) with `zarr-python`, which allows us to inspect the chunk and shard details.

In [ ]:
from pathlib import Path

import zarr

in_path = Path("./data/twophoton_series.zarr")

two_photon_series_zarr = zarr.open(in_path, mode="r")
print("overall shape of array", two_photon_series_zarr.shape)
print("shape of one chunk", two_photon_series_zarr.chunks)
print("total number of chunks", two_photon_series_zarr.nchunks)

As we have only used chunks here (no shards), the number of chunks == the number of files in the `/c` directory.

In [ ]:
chunks_dir = in_path / "c"

total_files = 0
for item in chunks_dir.glob("**/*"):
    if item.is_file():
        print(item)
        total_files += 1

print("Total files:", total_files)

## Chunks and shards

Let's read the same array into `dask`, so we can re-write it with different shards / chunks.

In [ ]:
import dask.array as da

two_photon_series = da.from_zarr(
    in_path, mode="r", chunks=two_photon_series_zarr.chunks
)
two_photon_series

Next, we loop over increasing shard sizes of (4508, 10, 5), (9016, 10, 5) and (18032, 10, 5), and for each we:

1. open a Zarr file on disk, specifying the shard size we want
2. write the array to disk chunk by chunk
3. record the total number of files written
4. record the total number of pixels in each shard

In [ ]:
import math

number_of_files = []
pixels_per_shard = []
shard_sizes = [(4508, 10, 5), (9016, 10, 5), (18032, 10, 5)]

for shard_size in shard_sizes:
    # step 1.
    shard_size_string = "-".join(str(size) for size in shard_size)
    outpath = Path(f"./data/two_photon_series_shards-{shard_size_string}.zarr")
    destination_on_disk = zarr.create_array(
        outpath,
        shape=two_photon_series.shape,
        dtype=two_photon_series.dtype,
        chunks=two_photon_series.chunksize,
        shards=shard_size,  # The shard size is set here
        zarr_format=3,
        overwrite=True,
    )

    # step 2.
    two_photon_series.to_zarr(
        destination_on_disk,
        compute=True,
        mode="w",
    )

    print("Files for shard size", shard_size)
    chunks_dir = outpath / "c"
    total_files = 0
    for item in chunks_dir.glob("**/*"):
        if item.is_file():
            print(item)
            total_files += 1

    number_of_files.append(total_files)
    pixels_per_shard.append(math.prod(shard_size))

We can now summarise our results in a plot:

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(pixels_per_shard, number_of_files)
plt.title("Shards vs number of files")
plt.xlabel("Pixels per shard")
plt.ylabel("Number of files")
plt.show()

From the graph, we can see that each time we double the shard size, we roughly half the total number of files.

## Stretch exercise

Modify the code above to compare shard size to read or write times. (use the code in `notebooks/5_remote_file_compression.ipynb` as a starting point)

Alternatively, feel free to try out some of the concepts presented here on your own data, invent your own related stretch exercise or help others in the room.

## Key take-aways

* Shards allow us to group multiple chunks together per file
* They help reduce the total number of files, while still allowing independent _read_ access to each chunk.
* Shards become the smallest unit we can write